# Diabetic Retinopathy Detection 👁️🏥
**End-to-End Deep Learning Pipeline**

This project aims to detect and classify Diabetic Retinopathy (DR) from retinal fundus images. The severity of DR is graded on a scale of 0 to 4:
- `0`: No DR
- `1`: Mild
- `2`: Moderate
- `3`: Severe
- `4`: Proliferative DR

### Pipeline Upgrades (From Basic to Professional):
1. **Proper Data Handling**: Custom PyTorch `Dataset` to lazily load images and avoid Out-Of-Memory (OOM) crashes.
2. **Advanced Augmentations**: Utilizing `Albumentations` for robust and fast computer vision augmentations.
3. **Imbalance Handling**: Using `WeightedRandomSampler` to handle underrepresented classes and achieve a higher Kappa score.
4. **Modern Architecture**: Transfer learning with `timm` (PyTorch Image Models) using `efficientnet_b3`.
5. **Robust Training**: Added Gradient accumulation, Learning Rate Scheduling (ReduceLROnPlateau), Early Stopping, and Quadratic Weighted Kappa tracking.
6. **Explainability**: Integrated Grad-CAM to visualize model attention maps.

## 1. Environment Setup & Imports

In [ ]:
!pip install -q timm albumentations grad-cam scikit-learn seaborn tqdm

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import cv2

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm

# Albumentations for Data Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration
Centralizing hyperparameters makes the notebook much easier to tune and experiment with.

In [ ]:
class CFG:
    seed = 42
    model_name = 'efficientnet_b3' # Switch to vit_base_patch16_224 or resnet50 easily
    img_size = 224
    epochs = 15
    batch_size = 32
    lr = 1e-4
    weight_decay = 1e-6
    num_classes = 5
    num_workers = 2
    accumulation_steps = 2 # Effective batch size = batch_size * accumulation_steps
    patience = 4
    
    # Paths (adjust according to your Google Drive setup)
    data_dir = "/content/data"
    zip_path = "/content/drive/MyDrive/aptos2019-blindness-detection.zip"
    
def seed_everything(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)

## 3. Data Extraction & Exploration

In [ ]:
# 1. Unzip the dataset efficiently if not already done
if not os.path.exists(CFG.data_dir):
    print("Extracting data...")
    if os.path.exists(CFG.zip_path):
        with zipfile.ZipFile(CFG.zip_path, 'r') as zip_ref:
            zip_ref.extractall(CFG.data_dir)
        print("Extraction complete.")
    else:
        print(f"Please ensure {CFG.zip_path} exists or update the path.")

# 2. Load DataFrames
TRAIN_CSV_PATH = os.path.join(CFG.data_dir, "train.csv")
TRAIN_IMAGE_DIR = os.path.join(CFG.data_dir, "train_images")

if os.path.exists(TRAIN_CSV_PATH):
    df = pd.read_csv(TRAIN_CSV_PATH)
    # Create absolute file paths
    df['file_path'] = df['id_code'].apply(lambda x: os.path.join(TRAIN_IMAGE_DIR, f"{x}.png"))
    
    # Split Data (80/20)
    train_df, val_df = train_test_split(
        df, 
        test_size=0.2, 
        random_state=CFG.seed, 
        stratify=df['diagnosis']
    )
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    
    print(f"Training set shape: {train_df.shape}")
    print(f"Validation set shape: {val_df.shape}")
else:
    print(f"Warning: {TRAIN_CSV_PATH} not found. Ensure dataset is mounted.")

In [ ]:
# EDA: Visualize Class Imbalance
if 'train_df' in locals():
    plt.figure(figsize=(10, 5))
    sns.countplot(data=train_df, x='diagnosis', palette='viridis')
    plt.title('Distribution of Diabetic Retinopathy Stages (Train Set)')
    plt.xlabel('Diagnosis (0: No DR, 1: Mild, 2: Moderate, 3: Severe, 4: Proliferative)')
    plt.ylabel('Count')
    plt.show()

## 4. Custom Dataset & Augmentations
Unlike loading everything into memory (which causes RAM crashes), we define a proper PyTorch `Dataset` that loads images lazily. We also replace standard transforms with `Albumentations` for advanced image augmentation.

In [ ]:
class RetinopathyDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.file_paths = df['file_path'].values
        self.labels = df['diagnosis'].values if 'diagnosis' in df.columns else None
        self.transforms = transforms
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_path = self.file_paths[index]
        image = cv2.imread(img_path)
        
        if image is None:
            # Fallback if image is missing/corrupted
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
            
        if self.labels is not None:
            label = torch.tensor(self.labels[index], dtype=torch.long)
            return image, label
        return image

In [ ]:
# Define Albumentations Pipelines
train_transforms = A.Compose([
    A.Resize(CFG.img_size, CFG.img_size),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=60, p=0.5),
    A.OneOf([
        A.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit=0.2, val_shift_limit=0.2, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    ], p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transforms = A.Compose([
    A.Resize(CFG.img_size, CFG.img_size),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [ ]:
if 'train_df' in locals():
    # Handling Class Imbalance with WeightedRandomSampler
    class_counts = train_df['diagnosis'].value_counts().sort_index().values
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[label] for label in train_df['diagnosis'].values]

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

    # Initialize Datasets & DataLoaders
    train_dataset = RetinopathyDataset(train_df, transforms=train_transforms)
    val_dataset = RetinopathyDataset(val_df, transforms=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, sampler=sampler, num_workers=CFG.num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, pin_memory=True)
    
    print("DataLoaders successfully initialized!")

## 5. Model Architecture
We modularize the model definition using `timm` so we can test different backbones.

In [ ]:
class RetinopathyModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        
    def forward(self, x):
        return self.model(x)

model = RetinopathyModel(CFG.model_name, CFG.num_classes)
model.to(device)
print(f"Model '{CFG.model_name}' successfully loaded!")

## 6. Training Pipeline
A clean, functional training loop. It features:
- **Quadratic Weighted Kappa (QWK)** evaluation.
- **Gradient Accumulation** to effectively train with a larger batch size on limited memory GPUs.
- **Early Stopping & Learning Rate Scheduling**.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []
    
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Training")
    for i, (images, labels) in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = criterion(outputs, labels) / CFG.accumulation_steps
        loss.backward()
        
        if (i + 1) % CFG.accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        running_loss += loss.item() * CFG.accumulation_steps * images.size(0)
        
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        preds_all.extend(preds)
        labels_all.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item() * CFG.accumulation_steps})
        
    epoch_loss = running_loss / len(dataloader.dataset)
    kappa = cohen_kappa_score(labels_all, preds_all, weights='quadratic')
    return epoch_loss, kappa

def valid_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []
    
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Validation")
    with torch.no_grad():
        for i, (images, labels) in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            preds_all.extend(preds)
            labels_all.extend(labels.cpu().numpy())
            
    epoch_loss = running_loss / len(dataloader.dataset)
    kappa = cohen_kappa_score(labels_all, preds_all, weights='quadratic')
    return epoch_loss, kappa, preds_all, labels_all

In [ ]:
if 'train_loader' in locals():
    best_kappa = -1.0
    patience_counter = 0

    history = {'train_loss': [], 'val_loss': [], 'train_kappa': [], 'val_kappa': []}

    for epoch in range(CFG.epochs):
        print(f"\nEpoch {epoch+1}/{CFG.epochs}")
        train_loss, train_kappa = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_kappa, val_preds, val_labels = valid_epoch(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_kappa'].append(train_kappa)
        history['val_kappa'].append(val_kappa)
        
        print(f"Train Loss: {train_loss:.4f} | Train QWK: {train_kappa:.4f}")
        print(f"Val Loss: {val_loss:.4f}   | Val QWK: {val_kappa:.4f}")
        
        scheduler.step(val_kappa)
        
        if val_kappa > best_kappa:
            best_kappa = val_kappa
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"🟢 Validation QWK improved to {best_kappa:.4f}. Model saved!")
            patience_counter = 0
        else:
            patience_counter += 1
            print(f"🟡 No improvement. Patience: {patience_counter}/{CFG.patience}")
            if patience_counter >= CFG.patience:
                print(f"🔴 Early stopping triggered after {epoch+1} epochs.")
                break

## 7. Evaluation & Analysis
Visualizing the model's convergence and generating the confusion matrix.

In [ ]:
if 'val_labels' in locals() and 'val_preds' in locals():
    # Plot Training History
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(history['train_loss'], label='Train Loss', marker='o')
    ax[0].plot(history['val_loss'], label='Val Loss', marker='o')
    ax[0].set_title('Loss Curve')
    ax[0].legend()

    ax[1].plot(history['train_kappa'], label='Train QWK', marker='o')
    ax[1].plot(history['val_kappa'], label='Val QWK', marker='o')
    ax[1].set_title('Quadratic Weighted Kappa (QWK) Curve')
    ax[1].legend()
    plt.show()
    
    # Confusion Matrix
    cm = confusion_matrix(val_labels, val_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Predicted Diagnosis')
    plt.ylabel('True Diagnosis')
    plt.title('Confusion Matrix')
    plt.show()
    
    # Detailed Report
    print("Classification Report:\n", classification_report(val_labels, val_preds))

## 8. Explainable AI with Grad-CAM
To build trust with medical applications, we use Grad-CAM to visualize where the model is "looking" to make its diagnosis. This highlights lesions, microaneurysms, and exudates in the eye.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

if 'train_df' in locals() and os.path.exists('best_model.pth'):
    # Load best weights
    model.load_state_dict(torch.load('best_model.pth'))
    model.eval()
    
    # Select Target layer for EfficientNet 
    target_layers = [model.model.conv_head]
    cam = GradCAM(model=model, target_layers=target_layers, use_cuda=torch.cuda.is_available())
    
    # Get a sample from Validation set
    sample_path = val_df['file_path'].iloc[5] # Change index to visualize other images
    true_label = val_df['diagnosis'].iloc[5]
    
    sample_img = cv2.imread(sample_path)
    if sample_img is not None:
        sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
        sample_img_resized = cv2.resize(sample_img, (CFG.img_size, CFG.img_size))
        rgb_img = np.float32(sample_img_resized) / 255.0
        
        # Transform for model input
        input_tensor = val_transforms(image=sample_img)['image'].unsqueeze(0).to(device)
        
        # Generate CAM
        targets = [ClassifierOutputTarget(true_label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
        
        # Overlay
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
        
        fig, ax = plt.subplots(1, 2, figsize=(12, 6))
        ax[0].imshow(rgb_img)
        ax[0].set_title(f"Original Image (True Class: {true_label})")
        ax[0].axis('off')
        
        ax[1].imshow(visualization)
        ax[1].set_title("Grad-CAM Attention Map")
        ax[1].axis('off')
        plt.show()
else:
    print("Run the training loop to generate 'best_model.pth' and visualize Grad-CAM.")